# タブラー特徴量への異常スコア追加検証

このノートブックでは、オートエンコーダーの再構成誤差を異常スコアとして計算し、
それをタブラー特徴量の1つとして追加した場合の効果を検証します。

## 目的
- オートエンコーダーで正常パターンを学習
- 再構成誤差を異常スコアとして計算
- 異常スコアをタブラー特徴量に追加
- 追加前後の分類性能を比較

## 期待される効果
- 異常な特徴量パターンの検出
- 分類精度の向上
- モデルの解釈性向上

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import pickle
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 8)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 1. データ読み込みと前処理

In [ ]:
# データパス設定
data_dir = Path("output/experiments/preprocess_check_ws64/preprocessed")

# データ読み込み
X_tabular = pickle.load(open(data_dir / "train_tabular.pkl", 'rb'))
y = pickle.load(open(data_dir / "train_labels.pkl", 'rb'))

# ラベル名
label_names = [
    'Above ear - pull hair', 'Cheek - pinch skin', 'Drink from bottle/cup',
    'Eyebrow - pull hair', 'Eyelash - pull hair', 'Feel around in tray and pull out an object',
    'Forehead - pull hairline', 'Forehead - scratch', 'Glasses on/off',
    'Neck - pinch skin', 'Neck - scratch', 'Pinch knee/leg skin',
    'Pull air toward your face', 'Scratch knee/leg skin', 'Text on phone',
    'Wave hello', 'Write name in air', 'Write name on leg'
]

print(f"=== データ情報 ===")
print(f"X_tabular shape: {X_tabular.shape}")
print(f"y shape: {y.shape}")
print(f"ラベル数: {len(np.unique(y))}")
print(f"サンプル数: {len(y)}")

# データの基本統計
print(f"\n=== 基本統計 ===")
print(f"平均値: {np.mean(X_tabular):.4f}")
print(f"標準偏差: {np.std(X_tabular):.4f}")
print(f"最小値: {np.min(X_tabular):.4f}")
print(f"最大値: {np.max(X_tabular):.4f}")
print(f"欠損値: {np.isnan(X_tabular).sum()}")

## 2. オートエンコーダー構築と学習

In [ ]:
class TabularAutoencoder(tf.keras.Model):
    def __init__(self, input_dim, latent_dim=64):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # エンコーダー
        self.encoder = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            
            tf.keras.layers.Dense(latent_dim, activation='relu')
        ])
        
        # デコーダー
        self.decoder = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            
            tf.keras.layers.Dense(256, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            
            tf.keras.layers.Dense(input_dim, activation='linear')
        ])
    
    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded
    
    def get_anomaly_score(self, x):
        """異常スコアを計算"""
        decoded, _ = self(x)
        reconstruction_error = tf.reduce_mean(tf.square(x - decoded), axis=1)
        return reconstruction_error

# オートエンコーダー構築
input_dim = X_tabular.shape[1]
latent_dim = 64

autoencoder = TabularAutoencoder(input_dim, latent_dim)
autoencoder.build((None, input_dim))

print(f"=== オートエンコーダー情報 ===")
print(f"入力次元: {input_dim}")
print(f"潜在次元: {latent_dim}")
print(f"圧縮率: {input_dim / latent_dim:.1f}:1")
print(f"総パラメータ数: {autoencoder.count_params():,}")

In [ ]:
# データ前処理
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_tabular)

# 学習データと検証データに分割
X_train, X_val = train_test_split(X_scaled, test_size=0.2, random_state=42)

print(f"=== データ前処理 ===")
print(f"正規化前 - 平均: {np.mean(X_tabular):.4f}, 標準偏差: {np.std(X_tabular):.4f}")
print(f"正規化後 - 平均: {np.mean(X_scaled):.4f}, 標準偏差: {np.std(X_scaled):.4f}")
print(f"学習データ: {X_train.shape}")
print(f"検証データ: {X_val.shape}")

In [ ]:
# オートエンコーダー学習
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

# コールバック設定
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6
    )
]

print("=== オートエンコーダー学習開始 ===")
history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

print("\n学習完了！")

## 3. 異常スコア計算と特徴量追加

In [ ]:
# 全データでの異常スコア計算
print("=== 異常スコア計算 ===")

anomaly_scores = autoencoder.get_anomaly_score(X_scaled).numpy()

print(f"異常スコア統計:")
print(f"  平均: {np.mean(anomaly_scores):.6f}")
print(f"  標準偏差: {np.std(anomaly_scores):.6f}")
print(f"  最小値: {np.min(anomaly_scores):.6f}")
print(f"  最大値: {np.max(anomaly_scores):.6f}")
print(f"  中央値: {np.median(anomaly_scores):.6f}")
print(f"  95パーセンタイル: {np.percentile(anomaly_scores, 95):.6f}")
print(f"  99パーセンタイル: {np.percentile(anomaly_scores, 99):.6f}")

In [ ]:
# 異常スコアの正規化
def normalize_anomaly_score(anomaly_scores, method='percentile'):
    """異常スコアを正規化"""
    if method == 'percentile':
        # パーセンタイルベースの正規化
        normalized = (anomaly_scores - np.percentile(anomaly_scores, 50)) / \
                     (np.percentile(anomaly_scores, 95) - np.percentile(anomaly_scores, 50) + 1e-8)
        normalized = np.clip(normalized, 0, 1)
    elif method == 'zscore':
        # Z-score正規化
        normalized = (anomaly_scores - np.mean(anomaly_scores)) / (np.std(anomaly_scores) + 1e-8)
        normalized = np.clip((normalized + 3) / 6, 0, 1)  # 0-1に変換
    elif method == 'minmax':
        # Min-Max正規化
        normalized = (anomaly_scores - np.min(anomaly_scores)) / \
                     (np.max(anomaly_scores) - np.min(anomaly_scores) + 1e-8)
    
    return normalized

# 様々な正規化方法を試す
normalization_methods = ['percentile', 'zscore', 'minmax']
normalized_scores = {}

for method in normalization_methods:
    normalized_scores[method] = normalize_anomaly_score(anomaly_scores, method)

print(f"=== 正規化結果 ===")
for method, scores in normalized_scores.items():
    print(f"{method}正規化:")
    print(f"  平均: {np.mean(scores):.4f}")
    print(f"  標準偏差: {np.std(scores):.4f}")
    print(f"  最小: {np.min(scores):.4f}")
    print(f"  最大: {np.max(scores):.4f}")
    print()

In [ ]:
# 特徴量に異常スコアを追加
recommended_normalized_scores = normalized_scores['percentile']  # 推奨方法

# 元の特徴量
X_original = X_tabular.copy()

# 異常スコアを追加した特徴量
X_enhanced = np.column_stack([X_tabular, recommended_normalized_scores])

print(f"=== 特徴量拡張結果 ===")
print(f"元の特徴量数: {X_original.shape[1]}")
print(f"拡張後の特徴量数: {X_enhanced.shape[1]}")
print(f"追加された特徴量: 異常スコア (正規化済み)")
print(f"データ形状: {X_enhanced.shape}")

# 拡張特徴量の基本統計
print(f"\n=== 拡張特徴量の統計 ===")
print(f"元特徴量 - 平均: {np.mean(X_original):.4f}, 標準偏差: {np.std(X_original):.4f}")
print(f"異常スコア - 平均: {np.mean(recommended_normalized_scores):.4f}, 標準偏差: {np.std(recommended_normalized_scores):.4f}")
print(f"拡張後 - 平均: {np.mean(X_enhanced):.4f}, 標準偏差: {np.std(X_enhanced):.4f}")

## 4. 分類性能比較

In [ ]:
# データ分割（分類タスク用）
X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    X_original, y, test_size=0.2, random_state=42, stratify=y
)

X_train_enh, X_test_enh, _, _ = train_test_split(
    X_enhanced, y, test_size=0.2, random_state=42, stratify=y
)

print(f"=== 分類タスク用データ分割 ===")
print(f"元特徴量 - 学習: {X_train_orig.shape}, テスト: {X_test_orig.shape}")
print(f"拡張特徴量 - 学習: {X_train_enh.shape}, テスト: {X_test_enh.shape}")
print(f"ラベル - 学習: {y_train.shape}, テスト: {y_test.shape}")

In [ ]:
# ランダムフォレストで分類性能比較
print("=== 分類性能比較 ===")

# 元の特徴量で学習
rf_original = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_original.fit(X_train_orig, y_train)
y_pred_orig = rf_original.predict(X_test_orig)
acc_orig = accuracy_score(y_test, y_pred_orig)

print(f"元特徴量での分類精度: {acc_orig:.4f}")

# 拡張特徴量で学習
rf_enhanced = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_enhanced.fit(X_train_enh, y_train)
y_pred_enh = rf_enhanced.predict(X_test_enh)
acc_enh = accuracy_score(y_test, y_pred_enh)

print(f"拡張特徴量での分類精度: {acc_enh:.4f}")
print(f"精度向上: {acc_enh - acc_orig:.4f} ({((acc_enh - acc_orig) / acc_orig * 100):.2f}%)")

In [ ]:
# 詳細な分類レポート
print("=== 詳細分類レポート ===")
print("\n元特徴量:")
print(classification_report(y_test, y_pred_orig, target_names=label_names))

print("\n拡張特徴量:")
print(classification_report(y_test, y_pred_enh, target_names=label_names))

In [ ]:
# 特徴量重要度の比較
print("=== 特徴量重要度比較 ===")

# 元の特徴量の重要度
feature_importance_orig = rf_original.feature_importances_
top_features_orig = np.argsort(feature_importance_orig)[-10:]

print(f"元特徴量の上位10個:")
for i, idx in enumerate(reversed(top_features_orig)):
    print(f"  {i+1:2d}. Feature {idx:3d}: {feature_importance_orig[idx]:.4f}")

# 拡張特徴量の重要度
feature_importance_enh = rf_enhanced.feature_importances_
top_features_enh = np.argsort(feature_importance_enh)[-10:]

print(f"\n拡張特徴量の上位10個:")
for i, idx in enumerate(reversed(top_features_enh)):
    if idx == X_original.shape[1]:  # 異常スコアの特徴量
        print(f"  {i+1:2d}. 異常スコア: {feature_importance_enh[idx]:.4f}")
    else:
        print(f"  {i+1:2d}. Feature {idx:3d}: {feature_importance_enh[idx]:.4f}")

# 異常スコアの重要度
anomaly_importance = feature_importance_enh[X_original.shape[1]]
print(f"\n異常スコアの重要度: {anomaly_importance:.4f}")
print(f"異常スコアの重要度順位: {np.sum(feature_importance_enh > anomaly_importance) + 1}/{len(feature_importance_enh)}")

## 5. 可視化と分析

In [ ]:
# 異常スコアの分布と分類性能の関係
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. 異常スコアの分布
axes[0, 0].hist(anomaly_scores, bins=50, alpha=0.7, color='lightblue', edgecolor='black')
axes[0, 0].set_title('異常スコアの分布')
axes[0, 0].set_xlabel('異常スコア')
axes[0, 0].set_ylabel('頻度')
axes[0, 0].axvline(np.percentile(anomaly_scores, 95), color='red', linestyle='--', 
                   label=f'95%: {np.percentile(anomaly_scores, 95):.4f}')
axes[0, 0].legend()

# 2. 正規化異常スコアの分布
axes[0, 1].hist(recommended_normalized_scores, bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('正規化異常スコアの分布')
axes[0, 1].set_xlabel('正規化異常スコア')
axes[0, 1].set_ylabel('頻度')

# 3. ラベル別異常スコア
label_anomaly_means = []
for label in np.unique(y):
    mask = y == label
    label_anomaly_means.append(np.mean(recommended_normalized_scores[mask]))

bars = axes[0, 2].bar(range(len(label_anomaly_means)), label_anomaly_means, alpha=0.7, color='lightcoral')
axes[0, 2].set_title('ラベル別平均異常スコア')
axes[0, 2].set_xlabel('ラベル')
axes[0, 2].set_ylabel('平均異常スコア')
axes[0, 2].set_xticks(range(len(label_anomaly_means)))
axes[0, 2].set_xticklabels([f'{i}\n{name[:10]}...' for i, name in enumerate(label_names)], 
                           rotation=45, ha='right')

# 4. 分類精度比較
methods = ['元特徴量', '拡張特徴量']
accuracies = [acc_orig, acc_enh]
bars = axes[1, 0].bar(methods, accuracies, alpha=0.7, color=['skyblue', 'lightgreen'])
axes[1, 0].set_title('分類精度比較')
axes[1, 0].set_ylabel('分類精度')
axes[1, 0].set_ylim(0, 1)

# 値のラベルを追加
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                    f'{acc:.3f}', ha='center', va='bottom')

# 5. 特徴量重要度比較
top_n = 10
orig_top = feature_importance_orig[top_features_orig[-top_n:]]
enh_top = feature_importance_enh[top_features_enh[-top_n:]]

x = np.arange(top_n)
width = 0.35

axes[1, 1].bar(x - width/2, orig_top, width, label='元特徴量', alpha=0.7, color='skyblue')
axes[1, 1].bar(x + width/2, enh_top, width, label='拡張特徴量', alpha=0.7, color='lightgreen')
axes[1, 1].set_title('上位10特徴量の重要度比較')
axes[1, 1].set_xlabel('特徴量順位')
axes[1, 1].set_ylabel('重要度')
axes[1, 1].legend()

# 6. 異常スコア vs 分類誤差
correct_predictions = (y_pred_enh == y_test)
test_anomaly_scores = recommended_normalized_scores[X_test_enh.shape[0]:]

axes[1, 2].scatter(test_anomaly_scores, correct_predictions, alpha=0.5, s=10)
axes[1, 2].set_title('異常スコア vs 分類正誤')
axes[1, 2].set_xlabel('異常スコア')
axes[1, 2].set_ylabel('正解 (1) / 不正解 (0)')

plt.tight_layout()
plt.show()

## 6. 結果の保存とまとめ

In [ ]:
# 結果の保存
output_dir = Path("output/experiments/anomaly_detection_results")
output_dir.mkdir(parents=True, exist_ok=True)

# オートエンコーダーを保存
autoencoder.save(output_dir / "tabular_autoencoder")

# 拡張特徴量を保存
with open(output_dir / "tabular_enhanced.pkl", 'wb') as f:
    pickle.dump(X_enhanced, f)

# 異常スコアを保存
with open(output_dir / "anomaly_scores.pkl", 'wb') as f:
    pickle.dump({
        'raw_scores': anomaly_scores,
        'normalized_scores': recommended_normalized_scores,
        'all_normalized_scores': normalized_scores
    }, f)

# 分類結果を保存
classification_results = {
    'original_accuracy': acc_orig,
    'enhanced_accuracy': acc_enh,
    'accuracy_improvement': acc_enh - acc_orig,
    'accuracy_improvement_percent': (acc_enh - acc_orig) / acc_orig * 100,
    'anomaly_score_importance': anomaly_importance,
    'anomaly_score_rank': np.sum(feature_importance_enh > anomaly_importance) + 1
}

with open(output_dir / "classification_results.json", 'w') as f:
    json.dump(classification_results, f, indent=2, default=str)

print(f"=== 結果保存完了 ===")
print(f"保存先: {output_dir}")
print(f"保存ファイル:")
print(f"  - tabular_autoencoder/ (オートエンコーダーモデル)")
print(f"  - tabular_enhanced.pkl (拡張特徴量)")
print(f"  - anomaly_scores.pkl (異常スコア)")
print(f"  - classification_results.json (分類結果)")

In [ ]:
# 分析結果の要約
print(f"=== 異常スコア特徴量追加の効果 ===")
print()

print(f"1. オートエンコーダー性能:")
print(f"   - 入力次元: {input_dim}")
print(f"   - 潜在次元: {latent_dim}")
print(f"   - 圧縮率: {input_dim / latent_dim:.1f}:1")
print(f"   - 最終学習損失: {history.history['loss'][-1]:.6f}")
print(f"   - 最終検証損失: {history.history['val_loss'][-1]:.6f}")
print()

print(f"2. 異常スコア統計:")
print(f"   - 平均: {np.mean(anomaly_scores):.6f}")
print(f"   - 標準偏差: {np.std(anomaly_scores):.6f}")
print(f"   - 95パーセンタイル: {np.percentile(anomaly_scores, 95):.6f}")
print(f"   - 99パーセンタイル: {np.percentile(anomaly_scores, 99):.6f}")
print()

print(f"3. 分類性能比較:")
print(f"   - 元特徴量精度: {acc_orig:.4f}")
print(f"   - 拡張特徴量精度: {acc_enh:.4f}")
print(f"   - 精度向上: {acc_enh - acc_orig:.4f} ({((acc_enh - acc_orig) / acc_orig * 100):.2f}%)")
print()

print(f"4. 異常スコアの重要度:")
print(f"   - 重要度: {anomaly_importance:.4f}")
print(f"   - 順位: {np.sum(feature_importance_enh > anomaly_importance) + 1}/{len(feature_importance_enh)}")
print()

print(f"5. 結論:")
if acc_enh > acc_orig:
    print(f"   ✅ 異常スコアの追加により分類精度が向上しました")
    print(f"   ✅ 異常スコアは重要な特徴量として認識されています")
else:
    print(f"   ❌ 異常スコアの追加による精度向上は見られませんでした")
    print(f"   ⚠️  他の特徴量エンジニアリング手法を検討してください")

print(f"\n異常スコアをタブラー特徴量として追加する準備が整いました！")